In [1]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from scipy import stats
from scipy.stats import spearmanr

## Cargamos los datos

In [8]:
df = pd.read_parquet(r"D:\DDAA\SimuladorEmpresarial\RRHH_220925_clean.parquet")
df.dtypes

ID                                  int64
Reason_absence             string[python]
Month_absence              string[python]
Day_week                   string[python]
Seasons                    string[python]
Transportation_expense            float64
Distance_Residence_Work           float64
Service_time                      float64
Age                               float64
Work_load_Average_day             float64
Hit_target                        float64
Disciplinary_failure                int64
Education                  string[python]
Son                                 int64
Social_drinker                      int64
Social_smoker                       int64
Pet                                 int64
Weight                            float64
Height                            float64
Body_mass_index                   float64
Absenteeism_hours                 float64
Education_numeric                   int64
Month_absence_order                 int64
Day_week_order                    

# Plan de análisis

## Objetivo del análisis 

Determinar qué características de los empleados están asociadas con un mayor desempeñoo laboral.

Para tener una medida de desempeño laboral consideraremos lo siguiente (para cada trabajador): el promedio de hit_target, la suma de horas de absentismo (o el recuento de absencias), si han tenido alguna falta disciplinaria o no, el número de faltas injustificadas. 

## ¿Cómo integrar toda esta información?

### Opción 1: clustering 

Podemos hacer clusters con estas variables y tendremos distintos grupos de trabajadores según su desempeño. A continuación, miraremos las características personales de cada grupo para intentar extraer conclusiones. 

### Opción 2: creación de medida de desempeño

En base a información sobre otras empresas, dar un peso a las distintas variables para obtener una medida de desempeño. 

## Respondiendo a la pregunta "¿Cómo afectan las catacterísticas personales al desempeño?"

### Análisis descriptivo

Ver qué características tienen los distintos clusters

### Modelo predictivo

Creamos un modelo con la combinación de características personales que nos permitan explicar en mayor medida la asignación de un trabajador a un cluster. 

Esto nos permitirá predecir a qué grupo de trabajadores pertenecerá un nuevo empleado en base a sus características personales.

# Cálculo de variables a utilizar en el clustering

In [9]:
pd.set_option("display.max.columns", None)
# Calculamos la media de hit target de cada trabajador y nos quedamos con un solo registro por trabajador 
df['Hit_target_avg'] = df.groupby('ID')['Hit_target'].transform('mean')

# Calculamos la media de Work_load_Average_day
df['Work_load_Average_day'] = df.groupby('ID')['Work_load_Average_day'].transform('mean')

# Calcular número d'absències per treballador
df['count_absenteeism'] = (
    (df['Absenteeism_hours'] != 0)  
    .groupby(df['ID'])              
    .transform('sum')               
)

# Calcular quantes absències injustificades té cada treballador
df["unjustified_absence"] = (
    (df["Reason_absence"] == 26)
    .groupby(df["ID"])
    .transform("sum")
)

# Suma total d'hores d'absència per treballador
df["sum_absenteeism_hours"] = df.groupby("ID")["Absenteeism_hours"].transform("sum")

# Ens quedem amb una única observació per treballador. Concretament l'última per si cap al final de l'any ha tingut alguna falta disciplinària (o han tingut un altre fill)
df = df.drop_duplicates(subset=["ID"], keep="last").reset_index(drop=True)

# Eliminem les variables que ja han perdut sentit
df = df.drop(columns=["Reason_absence","Month_absence", "Day_week", "Seasons","Hit_target","Absenteeism_hours","Education_numeric","Month_absence_order","Day_week_order","Seasons_order"])
df.head()

,ID,Transportation_expense,Distance_Residence_Work,Service_time,Age,Work_load_Average_day,Disciplinary_failure,Education,Son,Social_drinker,Social_smoker,Pet,Weight,Height,Body_mass_index,Hit_target_avg,count_absenteeism,unjustified_absence,sum_absenteeism_hours
0,6,189.0,29.0,13.0,33.0,274.829000,0,High school,2,0,0,2,69.0,167.0,25.0,94.875000,8,0,72.0
1,16,118.0,15.0,24.0,46.0,248.642500,0,High school,2,1,1,0,75.0,175.0,25.0,97.500000,2,0,16.0
2,25,235.0,16.0,8.0,32.0,254.817500,0,Postgraduate,0,0,0,0,75.0,178.0,25.0,95.600000,10,0,42.0
3,12,233.0,51.0,1.0,31.0,271.274000,0,Graduate,1,1,0,8,68.0,178.0,21.0,96.142857,7,0,34.0
4,27,184.0,42.0,7.0,27.0,299.956333,0,High school,0,0,0,0,58.0,167.0,21.0,95.166667,6,0,25.0


# Clustering

# Com varien les característiques personals a través dels diferents clusters?